# Анализ городских данных: практическое занятие по Pandas
## Разведочный анализ данных на примере датасета городских объектов

**Что мы будем делать:**
1. Загрузим и исследуем структуру данных
2. Оценим качество данных (пропуски, дубликаты, выбросы)
3. Проведём фильтрацию и группировку данных
4. Ответим на аналитические вопросы о городских объектах

## Часть 1. Подготовка окружения (5 минут)

Импортируем необходимые библиотеки и настраиваем отображение.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Настройка рабочего окружения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print('настойки завершены!')


настойки завершены!


## Часть 2. Загрузка и первичное знакомство с данными (10 минут)

Загрузим данные из JSON файла и посмотрим, что внутри.

In [ ]:
df = pd.read_json('city_for_practice.json')
df

,object_id,object_name,type,district,construction_year,area_sq_m,floors,last_renovation,needs_repair,has_accessible,served_residents,status
0,1,Торговый центр №52,residential,Западный,1986.00,987.20,NaN,2003.00,False,1.00,636,closed
1,2,Библиотека №15,education,Западный,1973.00,118.20,2.00,NaN,True,NaN,882,repair
2,3,Поликлиника №72,education,Южный,2012.00,98.80,1.00,2019.00,False,1.00,687,operational
3,4,Библиотека №21,education,Центральный,2007.00,494.90,1.00,2011.00,False,NaN,762,operational
4,5,Торговый центр №83,healthcare,Центральный,1975.00,NaN,1.00,NaN,False,0.00,784,operational
...,...,...,...,...,...,...,...,...,...,...,...,...
1495,1496,Поликлиника №34,retail,Восточный,1991.00,767.90,1.00,NaN,True,1.00,779,operational
1496,1497,Детский сад №89,education,Южный,1964.00,NaN,1.00,2011.00,False,0.00,832,operational
1497,1498,Детский сад №11,education,Северный,2022.00,69.10,2.00,2003.00,True,0.00,956,operational
1498,1499,Детский сад №31,education,Южный,1977.00,217.80,3.00,2014.00,True,0.00,653,operational


In [4]:
print(f'Форма датафрема: {df.shape}')
print(f'Количество строк: {df.shape[0]}')
print(f'Количество столбцов: {df.shape[1]}')
print('Название колонок')
df.columns

Форма датафрема: (1500, 12)
Количество строк: 1500
Количество столбцов: 12
Название колонок


Index(['object_id', 'object_name', 'type', 'district', 'construction_year',
       'area_sq_m', 'floors', 'last_renovation', 'needs_repair',
       'has_accessible', 'served_residents', 'status'],
      dtype='object')

In [ ]:
# тип данных
df.dtypes

object_id              int64
object_name           object
type                  object
district              object
construction_year    float64
area_sq_m            float64
floors               float64
last_renovation      float64
needs_repair            bool
has_accessible       float64
served_residents       int64
status                object
dtype: object

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   object_id          1500 non-null   int64  
 1   object_name        1500 non-null   object 
 2   type               1500 non-null   object 
 3   district           1500 non-null   object 
 4   construction_year  1433 non-null   float64
 5   area_sq_m          1450 non-null   float64
 6   floors             1469 non-null   float64
 7   last_renovation    1042 non-null   float64
 8   needs_repair       1500 non-null   bool   
 9   has_accessible     1343 non-null   float64
 10  served_residents   1500 non-null   int64  
 11  status             1380 non-null   object 
dtypes: bool(1), float64(5), int64(2), object(4)
memory usage: 142.1+ KB


### Вопросы:
- Сколько всего объектов в датасете?
- Какие типы данных встречаются?
- Какие столбцы могут содержать пропуски?

## Часть 3. Анализ качества данных (20 минут)

Проверим данные на наличие проблем: пропуски, дубликаты, некорректные значения.

In [11]:
# 3.1. Проверка на пропуски (null)
print('Количество пропусков в столбиках')
null_counts = df.isnull().sum()
null_counts[null_counts > 0]

Количество пропусков в столбиках


construction_year     67
area_sq_m             50
floors                31
last_renovation      458
has_accessible       157
status               120
dtype: int64

In [12]:
# Процент пропусков
null_percent = (df.isnull().sum() / len(df)) * 100
print('Процент пропусков')
null_percent[null_percent > 0]

Процент пропусков


construction_year    4.47
area_sq_m            3.33
floors               2.07
last_renovation     30.53
has_accessible      10.47
status               8.00
dtype: float64

In [ ]:
# 3.2. Проверка на дубликаты
print(f'Количество полных строк-дубликоатов: {df.duplicated().sum()}')
print(f'Количество дубликатов в object_name: {df['object_name'].duplicated().sum()}')

Количество полных строк-дубликоатов: 0
Количество дубликоатов в object_name: 829


In [18]:
# 3.3. Базовые статистики для числовых колонок
df.describe()

,object_id,construction_year,area_sq_m,floors,last_renovation,has_accessible,served_residents
count,1500.00,1433.00,1450.00,1469.00,1042.00,1343.00,1500.00
mean,750.50,1993.05,544.36,4.58,2012.11,0.60,747.40
std,433.16,18.81,481.99,4.57,7.24,0.49,149.62
min,1.00,1950.00,50.10,1.00,2000.00,0.00,437.00
25%,375.75,1979.00,196.55,2.00,2006.00,0.00,616.00
50%,750.50,1994.00,397.95,3.00,2012.00,1.00,745.00
75%,1125.25,2009.00,739.95,6.00,2018.00,1.00,876.00
max,1500.00,2024.00,3855.70,25.00,2024.00,1.00,1069.00


In [ ]:
# 3.4. Проверка на выбросы (например, очень большая площадь)


# Находим объекты с аномальной площадью (> 5000 кв.м)


## Часть 4. Анализ категориальных данных (20 минут)

Проанализируем распределение объектов по категориям.

In [19]:
# 4.1. Распределение по типам объектов
print('Распределение по типу объекта')
df['type'].value_counts()

Распределение по типу объекта


type
education      363
retail         321
residential    291
healthcare     235
sport          150
culture        140
Name: count, dtype: int64

In [21]:
# Процентное соотношение
print('Распределение по типу объекта в процентах')
df['type'].value_counts(normalize=True) * 100

Распределение по типу объекта в процентах


type
education     24.20
retail        21.40
residential   19.40
healthcare    15.67
sport         10.00
culture        9.33
Name: proportion, dtype: float64

In [22]:
# 4.2. Распределение по районам
print('Распределение по районам')
df['district'].value_counts()

Распределение по районам


district
Южный          312
Северный       306
Центральный    301
Восточный      232
Западный       220
Пригородный    129
Name: count, dtype: int64

In [23]:
# 4.3. Распределение по статусу объектов
print('Распределение по статусу')
df['status'].value_counts()

Распределение по статусу


status
operational    1116
repair          134
planned          70
closed           60
Name: count, dtype: int64

## Часть 5. Фильтрация и группировка данных (20 минут)

Научимся отбирать нужные строки и агрегировать данные.

In [28]:
df.columns

Index(['object_id', 'object_name', 'type', 'district', 'construction_year',
       'area_sq_m', 'floors', 'last_renovation', 'needs_repair',
       'has_accessible', 'served_residents', 'status'],
      dtype='object')

In [29]:
# 5.1. Фильтрация по условию
# Объекты, нуждающиеся в ремонте
need_repair = df[df['needs_repair'] == True]
print(f'Количество объектов под ремонт: {len(need_repair)}')
need_repair[['object_name', 'area_sq_m', 'status']]

Количество объектов под ремонт: 459


,object_name,area_sq_m,status
1,Библиотека №15,118.20,repair
7,Библиотека №24,1361.20,operational
8,Поликлиника №22,307.60,operational
11,Парк №2,717.50,operational
16,Школа №91,713.00,operational
...,...,...,...
1492,Спорткомплекс №23,454.90,planned
1493,Поликлиника №31,219.30,operational
1495,Поликлиника №34,767.90,operational
1497,Детский сад №11,69.10,operational


In [ ]:
# фильтрация по значению
df[df['district'].isin(['Центральный', 'Южный'])]

,object_id,object_name,type,district,construction_year,area_sq_m,floors,last_renovation,needs_repair,has_accessible,served_residents,status
2,3,Поликлиника №72,education,Южный,2012.00,98.80,1.00,2019.00,False,1.00,687,operational
3,4,Библиотека №21,education,Центральный,2007.00,494.90,1.00,2011.00,False,NaN,762,operational
4,5,Торговый центр №83,healthcare,Центральный,1975.00,NaN,1.00,NaN,False,0.00,784,operational
5,6,Торговый центр №75,education,Центральный,1986.00,204.90,3.00,NaN,False,1.00,567,operational
6,7,Поликлиника №88,retail,Центральный,1970.00,1198.60,6.00,2008.00,False,NaN,550,operational
...,...,...,...,...,...,...,...,...,...,...,...,...
1489,1490,Торговый центр №6,retail,Центральный,2010.00,436.20,2.00,2008.00,True,0.00,933,None
1492,1493,Спорткомплекс №23,education,Южный,1956.00,454.90,3.00,2021.00,True,1.00,632,planned
1493,1494,Поликлиника №31,retail,Южный,1986.00,219.30,1.00,2017.00,True,0.00,712,operational
1496,1497,Детский сад №89,education,Южный,1964.00,NaN,1.00,2011.00,False,0.00,832,operational


In [33]:
# 5.2. Сложные условия (И, ИЛИ)
# Объекты в Центральном районе, которым нужен ремонт
central_repair = df[(df['district'] == "Центральный") & (df['needs_repair'] == True)]
print(f'В центральном районе {len(central_repair)} зданий нуждаются в ремонте')
central_repair

В центральном районе 84 зданий нуждаются в ремонте


,object_id,object_name,type,district,construction_year,area_sq_m,floors,last_renovation,needs_repair,has_accessible,served_residents,status
35,36,Торговый центр №44,education,Центральный,1953.00,231.30,3.00,NaN,True,NaN,944,operational
47,48,Парк №14,healthcare,Центральный,1994.00,72.90,2.00,2015.00,True,0.00,512,operational
67,68,Поликлиника №5,residential,Центральный,1998.00,708.90,6.00,NaN,True,0.00,732,operational
79,80,Библиотека №47,education,Центральный,1966.00,858.30,4.00,2021.00,True,0.00,628,operational
150,151,Детский сад №67,education,Центральный,1994.00,833.60,2.00,2014.00,True,0.00,771,operational
...,...,...,...,...,...,...,...,...,...,...,...,...
1440,1441,Спорткомплекс №97,residential,Центральный,1993.00,345.40,1.00,NaN,True,0.00,666,operational
1458,1459,Жилой дом №30,retail,Центральный,2009.00,1089.90,2.00,2012.00,True,0.00,879,operational
1468,1469,Детский сад №10,healthcare,Центральный,2009.00,354.30,4.00,2010.00,True,0.00,534,operational
1469,1470,Школа №66,healthcare,Центральный,1982.00,NaN,20.00,2009.00,True,0.00,638,operational


In [34]:
df.columns

Index(['object_id', 'object_name', 'type', 'district', 'construction_year',
       'area_sq_m', 'floors', 'last_renovation', 'needs_repair',
       'has_accessible', 'served_residents', 'status'],
      dtype='object')

In [36]:
# 5.3. Группировка данных
# Средняя площадь объектов по типам
df.groupby('type')['area_sq_m'].mean()

type
culture       490.69
education     550.57
healthcare    508.13
residential   559.36
retail        561.08
sport         570.23
Name: area_sq_m, dtype: float64

In [37]:
# Количество объектов по типам и районам (сводная таблица)
pivot = pd.crosstab(df['type'], df['district'])
pivot

district,Восточный,Западный,Пригородный,Северный,Центральный,Южный
type,,,,,,
culture,34,18,15,27,21,25
education,55,45,29,77,82,75
healthcare,35,35,21,45,51,48
residential,33,49,27,60,54,68
retail,51,52,27,63,61,67
sport,24,21,10,34,32,29


In [38]:
df.columns

Index(['object_id', 'object_name', 'type', 'district', 'construction_year',
       'area_sq_m', 'floors', 'last_renovation', 'needs_repair',
       'has_accessible', 'served_residents', 'status'],
      dtype='object')

In [39]:
# 5.4. Множественная агрегация
agg_result = df.groupby('district').agg({
        'area_sq_m' : ['sum', 'min', 'max'],
        'served_residents' : 'sum',
        'needs_repair' : lambda x: x.sum()     
})
agg_result

area_sq_m               served_residents needs_repair
                  sum   min     max              sum     <lambda>
district                                                         
Восточный   119980.20 53.50 3341.90           171331           72
Западный    116359.20 51.30 3855.70           166157           69
Пригородный  71447.20 50.50 2299.30            95857           31
Северный    166580.20 50.10 2753.70           233560          102
Центральный 161879.40 51.70 2439.40           222110           84
Южный       153072.70 50.50 3136.50           232086          101

## Часть 6. Работа с пропусками (10 минут)

Научимся обрабатывать пропущенные значения.

In [40]:
# 6.1. Варианты заполнения пропусков
# Создаём копию для экспериментов
df_test = df.copy()

In [ ]:
# Заполняем пропуски в 'construction_year' медианным значением
med_year = df['construction_year'].median()
df_test['construction_year'] = df_test['construction_year'].fillna(med_year)

In [45]:
# Заполняем пропуски в 'status' значением 'unknown'
df_test['status'] = df['status'].fillna('unknown')

In [47]:
df_test['status'].value_counts()

status
operational    1116
repair          134
unknown         120
planned          70
closed           60
Name: count, dtype: int64

In [50]:
df_test['construction_year'].isna().sum()

np.int64(0)

In [41]:
print('Количество пропусков в столбиках')
null_counts = df.isnull().sum()
null_counts[null_counts > 0]

Количество пропусков в столбиках


construction_year     67
area_sq_m             50
floors                31
last_renovation      458
has_accessible       157
status               120
dtype: int64

In [51]:
# 6.2. Удаление строк с пропусками (если нужно)
df_test.dropna(subset=['area_sq_m', 'floors'], inplace=True)
print(f'Было {len(df)}, Стало: {len(df_test)}')

Было 1500, Стало: 1420


## Часть 7. Совместная работа: исследуем данные вместе (15 минут)

Давайте вместе ответим на несколько вопросов о наших данных. 

---

### 🔍 Вопрос 1 (3 минуты)

**Какой тип объектов (`type`) встречается чаще всего? А какой реже всего?**

In [ ]:
# Попробуйте написать код здесь


**Подсказка:** используйте `value_counts()`

### 🔍 Вопрос 2 (3 минуты)

**В каком районе больше всего объектов, нуждающихся в ремонте?**

In [53]:
# Ваш код
df[df['needs_repair'] == True]['district'].value_counts()

district
Северный       102
Южный          101
Центральный     84
Восточный       72
Западный        69
Пригородный     31
Name: count, dtype: int64

**Подсказка:** отфильтруйте `needs_repair == True`, затем сгруппируйте по району

### 🔍 Вопрос 3 (3 минуты)

**Сколько в среднем жителей обслуживает один спортивный объект (`sport`) и один медицинский (`healthcare`)?**

In [57]:
# Ваш код
df_resedent = df[df['type'].isin(['sport', 'healthcare'])]
df_resedent.groupby('type')['served_residents'].mean().round()

type
healthcare   743.00
sport        741.00
Name: served_residents, dtype: float64

**Подсказка:** используйте группировку по `type` и агрегацию `mean` по `served_residents`

### 🔍 Вопрос 4 (3 минуты)

**Какой район лидирует по суммарной площади всех объектов?**

In [ ]:
# Ваш код


**Подсказка:** сгруппируйте по району и просуммируйте `area_sq_m`

### 🔍 Вопрос 5 (3 минуты)

**Есть ли связь между годом постройки и необходимостью ремонта? (Сравните средний год постройки у объектов, которым нужен ремонт, и у тех, кому не нужен)**

In [ ]:
# Ваш код


**Подсказка:** сгруппируйте по `needs_repair` и возьмите среднее `construction_year`

### Обсуждение результатов

Давайте вместе разберём, что получилось:
- Какие выводы можно сделать из наших расчётов?
- Какие районы требуют большего внимания?
- Какие типы объектов наиболее востребованы?

## Часть 8. Продвинутые методы анализа (дополнительный модуль, 30 минут)

Давайте научимся создавать новые признаки и выполнять сложные группировки.


### 8.1. Создание категориальных признаков (10 минут)

Создадим новые колонки на основе года постройки.

In [ ]:
# 8.1.1. Создаём колонку 'age' — возраст объекта


# 8.1.2. Создаём категории по возрасту


In [ ]:
# 8.1.3. Анализ распределения по возрастным категориям


### 8.2. Создание новых признаков (feature engineering) (10 минут)

Создадим метрику эффективности использования площади.

In [ ]:
# 8.2.1. Плотность населения на единицу площади


# 8.2.2. Категория загруженности


In [ ]:
# 8.2.3. Анализ загруженности по типам объектов


# .unstack() Превращаем категории загруженности (low, medium, high) в отдельные столбцы



### 8.3. Продвинутая фильтрация и сортировка (10 минут)

Научимся находить топ-объекты по разным критериям.

In [ ]:
# 8.3.1. Топ-5 объектов по обслуживаемым жителям


In [ ]:
# 8.3.2. Топ-3 самых старых объекта в каждом районе


In [ ]:
# 8.3.3. Объекты, требующие срочного внимания (ремонт + старая постройка + высокая загруженность)


### Практическое задание (для самостоятельной работы)

1. Создайте колонку `'efficiency'` — отношение количества жителей к площади (уже сделали).
2. Найдите район с самой высокой средней плотностью населения.
3. Найдите объекты, которые обслуживают больше 1000 жителей, но имеют площадь меньше 200 кв.м.
4. Постройте рейтинг типов объектов по среднему возрасту (от самых старых к самым новым).

In [ ]:
# Решения: 2.


In [ ]:
# 3.


In [ ]:
# 4.
